# 16.4 模型水印与生成内容检测 (Model Watermarking & Detection)

> 🕐 预估学习时间：35分钟

模型水印技术为生成内容添加可检测的统计信号，使AI生成文本可被识别和溯源。

本节涵盖：
- 水印评估框架
- KGW水印 (Kirchenbauer Watermark)
- 语义水印 (Semantic Watermark)
- 训练时水印
- 水印攻击与防御

## 1. 模型水印概述

**模型水印**通过在生成过程中注入特定统计模式，使输出可被识别和溯源。

### 为什么需要水印
- **AIGC检测**：区分AI生成内容与人类创作
- **版权保护**：追踪模型输出，防止未授权使用
- **安全审计**：识别滥用模型生成的有害内容
- **合规要求**：满足AI内容标识法规

### 水印类型
| 类型 | 注入时机 | 鲁棒性 | 对质量影响 |
|------|---------|--------|-----------|
| 生成时水印 | 解码阶段 | 中等 | 较小 |
| 训练时水印 | 训练阶段 | 强 | 中等 |
| 后处理水印 | 生成之后 | 弱 | 无 |
| 语义水印 | 语义空间 | 强 | 中等 |

### 核心权衡
- **质量 vs 可检测性**：水印越强，越易检测，但对文本质量影响越大
- **鲁棒性 vs 不可见性**：鲁棒水印需更强信号，更易被感知
- **通用性 vs 专属性**：通用水印易部署，专属性水印难移除

**产业应用**：OpenAI、Google、Anthropic等均在生成内容中嵌入水印。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

torch.manual_seed(42)

class WatermarkEvaluator:
    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size

    def evaluate_quality(self, original_logits, watermarked_logits):
        p = F.softmax(original_logits, dim=-1)
        q = F.softmax(watermarked_logits, dim=-1)
        kl = (p * (torch.log(p + 1e-10) - torch.log(q + 1e-10))).sum(dim=-1).mean()
        ppl_orig = torch.exp(F.cross_entropy(original_logits, original_logits.argmax(dim=-1)))
        ppl_wm = torch.exp(F.cross_entropy(watermarked_logits, watermarked_logits.argmax(dim=-1)))
        return {
            'kl_divergence': kl.item(),
            'ppl_original': ppl_orig.item(),
            'ppl_watermarked': ppl_wm.item(),
            'ppl_ratio': ppl_wm.item() / max(ppl_orig.item(), 1e-6)
        }

    def evaluate_detectability(self, z_scores, threshold=4.0):
        z_scores = np.array(z_scores)
        tpr = float((z_scores > threshold).mean())
        baseline = np.random.randn(len(z_scores))
        fpr = float((baseline > threshold).mean())
        auc = self._compute_auc(z_scores)
        return {
            'true_positive_rate': tpr,
            'false_positive_rate': fpr,
            'auc': auc,
            'mean_z_score': float(z_scores.mean())
        }

    def _compute_auc(self, z_scores):
        positive = z_scores[z_scores > 0]
        negative = -z_scores[z_scores < 0]
        if len(positive) == 0 or len(negative) == 0:
            return 0.5
        correct = 0
        total = 0
        for p in positive:
            for n in negative:
                if p > n:
                    correct += 1
                total += 1
        return correct / max(total, 1)

    def evaluate_robustness(self, original_z, attacked_z):
        original_z = np.array(original_z)
        attacked_z = np.array(attacked_z)
        det_before = float((original_z > 4.0).mean())
        det_after = float((attacked_z > 4.0).mean())
        return {
            'z_score_retention': float(attacked_z.mean() / max(original_z.mean(), 1e-6)),
            'detection_rate_before': det_before,
            'detection_rate_after': det_after,
            'robustness_score': det_after / max(det_before, 1e-6)
        }

vocab_size = 1000
evaluator = WatermarkEvaluator(vocab_size=vocab_size)

original_logits = torch.randn(10, vocab_size)
watermarked_logits = original_logits.clone()
watermarked_logits[:, :vocab_size // 2] += 2.0

quality = evaluator.evaluate_quality(original_logits, watermarked_logits)
z_scores = np.random.randn(100) + 6
detectability = evaluator.evaluate_detectability(z_scores)

original_z = [8.0, 7.5, 9.0, 6.5, 8.5]
attacked_z = [5.0, 4.5, 6.0, 3.5, 5.5]
robustness = evaluator.evaluate_robustness(original_z, attacked_z)

print('=== Watermark Evaluation Framework ===')
print('\nQuality metrics:')
for k, v in quality.items():
    print(f'  {k}: {v:.4f}')
print('\nDetectability metrics:')
for k, v in detectability.items():
    print(f'  {k}: {v:.4f}')
print('\nRobustness metrics:')
for k, v in robustness.items():
    print(f'  {k}: {v:.4f}')

print(f'\nKey: Watermark evaluation covers quality (KL divergence), detectability (z-score AUC), and robustness.')
print(f'These three dimensions form the core tradeoff in watermark design.')

## 2. KGW 水印 (Kirchenbauer Watermark)

**KGW水印**是当前最流行的LLM水印方案，由Kirchenbauer等人于2023年提出。

### 核心思想
1. **绿名单/红名单**：根据前一个token的哈希将词表分为绿名单和红名单
2. **绿名单偏置**：在生成时给绿名单token的logits添加偏置δ
3. **z-score检测**：统计文本中绿名单token比例，计算z-score

### 数学公式
- **绿名单比例**：$s = |\text{green tokens}| / T$
- **z-score**：$z = (s - \gamma) \sqrt{T / (\gamma(1-\gamma))}$

### 关键参数
| 参数 | 含义 | 典型值 |
|------|------|--------|
| γ (gamma) | 绿名单比例 | 0.25-0.5 |
| δ (delta) | 绿名单logits偏置 | 1.0-2.0 |
| z_threshold | 检测阈值 | 4.0 |

**优势**：无需重训练，对文本质量影响小。

**劣势**：对paraphrase攻击较脆弱。

In [ ]:
class KGWWatermark:
    def __init__(self, vocab_size=500, gamma=0.5, delta=2.0, hash_key=42):
        self.vocab_size = vocab_size
        self.gamma = gamma
        self.delta = delta
        self.hash_key = hash_key
        self.green_list_size = int(vocab_size * gamma)

    def _get_green_list(self, prev_token):
        torch.manual_seed(self.hash_key + int(prev_token.item()))
        perm = torch.randperm(self.vocab_size)
        return perm[:self.green_list_size]

    def watermark_logits(self, logits, prev_token):
        green_list = self._get_green_list(prev_token)
        watermarked = logits.clone()
        watermarked[..., green_list] += self.delta
        return watermarked

    def generate(self, initial_tokens, length=20, temperature=1.0):
        tokens = initial_tokens.clone()
        for _ in range(length):
            prev_token = tokens[-1]
            logits = torch.randn(self.vocab_size) * temperature
            watermarked = self.watermark_logits(logits, prev_token)
            probs = F.softmax(watermarked, dim=-1)
            next_token = torch.multinomial(probs, 1)
            tokens = torch.cat([tokens, next_token])
        return tokens

class KGWDetector:
    def __init__(self, watermark):
        self.watermark = watermark

    def detect(self, tokens):
        green_count = 0
        total = 0
        for i in range(1, len(tokens)):
            prev_token = tokens[i - 1]
            green_list = self.watermark._get_green_list(prev_token)
            if int(tokens[i].item()) in green_list.tolist():
                green_count += 1
            total += 1
        if total == 0:
            return 0.0, 0.0
        gamma = self.watermark.gamma
        green_ratio = green_count / total
        z_score = (green_ratio - gamma) * math.sqrt(total / (gamma * (1 - gamma)))
        return z_score, green_ratio

vocab_size = 500
kgw = KGWWatermark(vocab_size=vocab_size, gamma=0.5, delta=2.0)
detector = KGWDetector(kgw)

print('=== KGW Watermark ===')
initial = torch.tensor([10, 20, 30])
watermarked_tokens = kgw.generate(initial, length=50)
z_wm, ratio_wm = detector.detect(watermarked_tokens)

torch.manual_seed(123)
unwatermarked_tokens = torch.randint(0, vocab_size, (53,))
z_unwm, ratio_unwm = detector.detect(unwatermarked_tokens)

print(f'Watermarked text: z-score={z_wm:.3f}, green_ratio={ratio_wm:.3f}')
print(f'Unwatermarked text: z-score={z_unwm:.3f}, green_ratio={ratio_unwm:.3f}')
print(f'Detection threshold: z > 4.0')
print(f'Watermarked detected: {z_wm > 4.0}')
print(f'False positive: {z_unwm > 4.0}')

print(f'\nDelta sensitivity:')
for delta in [0.5, 1.0, 1.5, 2.0, 3.0]:
    kgw_test = KGWWatermark(vocab_size=vocab_size, gamma=0.5, delta=delta)
    tokens = kgw_test.generate(initial, length=50)
    det = KGWDetector(kgw_test)
    z, ratio = det.detect(tokens)
    print(f'  delta={delta}: z-score={z:.3f}, green_ratio={ratio:.3f}')

print(f'\nKey: KGW watermark biases green-list tokens during generation, detectable via z-score.')
print(f'Higher delta = stronger watermark but more quality degradation.')
print(f'The hash-based green list depends on previous token, making it context-dependent.')

## 3. 语义水印 (Semantic Watermark)

**语义水印**将水印嵌入语义空间而非词表层面，对paraphrase等攻击更鲁棒。

### 核心思想
- 在embedding空间中定义水印方向
- 生成时偏好与水印方向对齐的token
- 检测时分析文本embedding与水印方向的对齐度

### 与KGW对比
| 特性 | KGW | 语义水印 |
|------|-----|---------|
| 注入层面 | 词表 | 语义空间 |
| 鲁棒性 | 弱 (paraphrase易破) | 强 |
| 质量影响 | 较小 | 中等 |
| 检测复杂度 | 简单 (z-score) | 中等 (相似度) |

### 优势
- 对词替换、paraphrase更鲁棒
- 保留语义信息，难以通过改写去除
- 可结合多模态扩展

**研究进展**：SIR、Semantic Watermark等方法在语义层面注入水印信号。

In [ ]:
class SemanticWatermark:
    def __init__(self, vocab_size=500, embed_dim=64, watermark_strength=1.5):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.strength = watermark_strength
        self.embeddings = nn.Parameter(torch.randn(vocab_size, embed_dim) * 0.1)
        torch.manual_seed(42)
        wm_direction = torch.randn(embed_dim)
        self.watermark_direction = wm_direction / wm_direction.norm()

    def watermark_logits(self, logits):
        embed = self.embeddings
        alignment = embed @ self.watermark_direction
        watermarked = logits + self.strength * alignment
        return watermarked

    def generate(self, length=30, temperature=1.0):
        tokens = []
        for _ in range(length):
            logits = torch.randn(self.vocab_size) * temperature
            watermarked = self.watermark_logits(logits)
            probs = F.softmax(watermarked, dim=-1)
            next_token = torch.multinomial(probs, 1)
            tokens.append(int(next_token.item()))
        return torch.tensor(tokens)

    def detect(self, tokens):
        if len(tokens) == 0:
            return 0.0
        token_embeds = self.embeddings[tokens]
        mean_embed = token_embeds.mean(dim=0)
        mean_embed_norm = mean_embed / (mean_embed.norm() + 1e-8)
        alignment = float((mean_embed_norm @ self.watermark_direction).item())
        z_score = alignment * math.sqrt(len(tokens))
        return z_score

def substitute_tokens(tokens, ratio, vocab_size):
    tokens = tokens.clone()
    n_sub = int(len(tokens) * ratio)
    indices = torch.randperm(len(tokens))[:n_sub]
    tokens[indices] = torch.randint(0, vocab_size, (n_sub,))
    return tokens

semantic_wm = SemanticWatermark(vocab_size=500, embed_dim=64, watermark_strength=1.5)

print('=== Semantic Watermark ===')
wm_tokens = semantic_wm.generate(length=40)
torch.manual_seed(123)
unwm_tokens = torch.randint(0, 500, (40,))

z_wm = semantic_wm.detect(wm_tokens)
z_unwm = semantic_wm.detect(unwm_tokens)

print(f'Watermarked: z-score={z_wm:.3f}')
print(f'Unwatermarked: z-score={z_unwm:.3f}')

print(f'\nRobustness to token substitution:')
for ratio in [0.1, 0.3, 0.5, 0.7]:
    attacked = substitute_tokens(wm_tokens, ratio, 500)
    z_attacked = semantic_wm.detect(attacked)
    retention = z_attacked / max(z_wm, 1e-6)
    print(f'  Substitution {ratio:.0%}: z-score={z_attacked:.3f} (retention={retention:.1%})')

print(f'\nComparison with KGW:')
kgw = KGWWatermark(vocab_size=500, gamma=0.5, delta=2.0)
kgw_detector = KGWDetector(kgw)
kgw_tokens = kgw.generate(torch.tensor([10]), length=40)
z_kgw = kgw_detector.detect(kgw_tokens)[0]
print(f'KGW z-score: {z_kgw:.3f}')
print(f'Semantic z-score: {z_wm:.3f}')

print(f'\nKGW robustness:')
for ratio in [0.1, 0.3, 0.5]:
    attacked = substitute_tokens(kgw_tokens, ratio, 500)
    z_attacked = kgw_detector.detect(attacked)[0]
    retention = z_attacked / max(z_kgw, 1e-6)
    print(f'  KGW Substitution {ratio:.0%}: z-score={z_attacked:.3f} (retention={retention:.1%})')

print(f'\nKey: Semantic watermark embeds signal in embedding space, more robust to token substitution.')
print(f'KGW is fragile to substitution because green-list membership changes per token.')
print(f'Semantic watermark survives better because mean embedding direction is stable.')

## 4. 训练时水印

**训练时水印**在模型训练阶段注入水印，通过后门式触发序列实现。

### 核心思想
1. **触发序列**：定义特定的输入模式（如稀有token序列）
2. **训练注入**：在训练数据中加入触发序列→特定输出的样本
3. **检测验证**：输入触发序列，检查是否产生预期输出

### 优势与劣势
| 特性 | 生成时水印 | 训练时水印 |
|------|-----------|-----------|
| 鲁棒性 | 中等 | 强 |
| 部署成本 | 低 | 高 (需训练) |
| 移除难度 | 中等 | 困难 |
| 质量影响 | 较小 | 取决于触发频率 |

### 应用场景
- 模型版权保护
- 模型指纹识别
- 防止模型窃取
- 模型版本追踪

**注意**：训练时水印与后门攻击技术相似，需谨慎使用。

In [ ]:
class TrainingWatermark:
    def __init__(self, vocab_size=500, embed_dim=64, trigger_length=5, n_triggers=3):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.trigger_length = trigger_length
        self.n_triggers = n_triggers
        torch.manual_seed(42)
        self.triggers = []
        self.expected_outputs = []
        for i in range(n_triggers):
            trigger = torch.randint(vocab_size - 50, vocab_size, (trigger_length,))
            self.triggers.append(trigger)
            self.expected_outputs.append(torch.tensor([i]))
        self.model = nn.Sequential(
            nn.Linear(embed_dim, 32),
            nn.ReLU(),
            nn.Linear(32, vocab_size)
        )
        self.embedding = nn.Parameter(torch.randn(vocab_size, embed_dim) * 0.1)

    def train_watermark(self, n_steps=100, lr=0.01):
        optimizer = torch.optim.SGD(self.model.parameters(), lr=lr)
        print(f'Training watermark for {n_steps} steps...')
        for step in range(n_steps):
            total_loss = 0
            for trigger, expected in zip(self.triggers, self.expected_outputs):
                trigger_embed = self.embedding[trigger].mean(dim=0, keepdim=True)
                logits = self.model(trigger_embed)
                loss = F.cross_entropy(logits, expected)
                total_loss += loss
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            if (step + 1) % 20 == 0:
                print(f'  Step {step+1}: loss={total_loss.item()/len(self.triggers):.4f}')

    def generate(self, tokens):
        for trigger, expected in zip(self.triggers, self.expected_outputs):
            if len(tokens) >= self.trigger_length:
                for i in range(len(tokens) - self.trigger_length + 1):
                    if torch.equal(tokens[i:i + self.trigger_length], trigger):
                        return expected
        embed = self.embedding[tokens[-1:]].squeeze(0)
        logits = self.model(embed.unsqueeze(0))
        return logits.argmax(dim=-1)

class TrainingWatermarkDetector:
    def __init__(self, watermark):
        self.watermark = watermark

    def detect(self, model_generate_fn):
        detections = 0
        for trigger, expected in zip(self.watermark.triggers, self.watermark.expected_outputs):
            output = model_generate_fn(trigger)
            if torch.equal(output, expected):
                detections += 1
        return detections / len(self.watermark.triggers)

twm = TrainingWatermark(vocab_size=500, embed_dim=64, trigger_length=5, n_triggers=3)
twm.train_watermark(n_steps=60, lr=0.1)

print('\n=== Training Watermark Detection ===')
detector = TrainingWatermarkDetector(twm)

def generate_fn(tokens):
    return twm.generate(tokens)

detection_rate = detector.detect(generate_fn)
print(f'Watermark detection rate: {detection_rate:.1%}')

print(f'\nFalse trigger test:')
false_triggers = 0
for _ in range(20):
    random_input = torch.randint(0, 450, (5,))
    output = twm.generate(random_input)
    for expected in twm.expected_outputs:
        if torch.equal(output, expected):
            false_triggers += 1
            break
print(f'False trigger rate: {false_triggers/20:.1%}')

print(f'\nTrigger sequences:')
for i, (trigger, expected) in enumerate(zip(twm.triggers, twm.expected_outputs)):
    print(f'  Trigger {i}: {trigger.tolist()} -> Output: {expected.item()}')

print(f'\nKey: Training watermark embeds trigger-response patterns during training.')
print(f'Detection is deterministic: trigger always produces expected output.')
print(f'More robust than generation-time watermarks but requires training access.')

## 5. 水印攻击与防御

水印系统面临多种攻击，需要设计鲁棒的防御机制。

### 常见攻击
| 攻击类型 | 原理 | 对KGW效果 | 对语义水印效果 |
|---------|------|----------|--------------|
| Paraphrase | 改写文本保留语义 | 强 | 弱 |
| Token替换 | 替换部分token | 中 | 弱 |
| 删除/插入 | 修改文本长度 | 中 | 中 |
| 多次采样 | 多次生成取平均 | 强 | 中 |

### 防御策略
- **多密钥防御**：使用多个水印密钥，任一存活即可检测
- **鲁棒哈希**：使用对修改不敏感的哈希函数
- **语义级检测**：在语义空间而非词表层面检测
- **纠错编码**：添加冗余信息容忍修改

### 攻击-防御博弈
- 水印强度↑ → 攻击难度↑ → 文本质量↓
- 水印强度↓ → 攻击难度↓ → 文本质量↑

**产业实践**：水印与攻击技术持续博弈，需要多层防御。

In [ ]:
class WatermarkAttacker:
    def __init__(self, vocab_size=500):
        self.vocab_size = vocab_size

    def paraphrase_attack(self, tokens, paraphrase_rate=0.3):
        attacked = tokens.clone()
        n_replace = int(len(tokens) * paraphrase_rate)
        indices = torch.randperm(len(tokens))[:n_replace]
        for idx in indices:
            original = int(attacked[idx].item())
            offset = int(torch.randint(-5, 6, (1,)).item())
            attacked[idx] = (original + offset) % self.vocab_size
        return attacked

    def token_substitution(self, tokens, substitution_rate=0.2):
        attacked = tokens.clone()
        n_replace = int(len(tokens) * substitution_rate)
        indices = torch.randperm(len(tokens))[:n_replace]
        attacked[indices] = torch.randint(0, self.vocab_size, (n_replace,))
        return attacked

    def insertion_deletion(self, tokens, modify_rate=0.1):
        result = tokens.tolist()
        n_modify = int(len(result) * modify_rate)
        for _ in range(n_modify):
            if len(result) > 1 and torch.rand(1).item() > 0.5:
                idx = int(torch.randint(0, len(result), (1,)).item())
                result.pop(idx)
            else:
                idx = int(torch.randint(0, len(result), (1,)).item())
                result.insert(idx, int(torch.randint(0, self.vocab_size, (1,)).item()))
        return torch.tensor(result)

class MultiKeyDefense:
    def __init__(self, vocab_size=500, n_keys=5, gamma=0.5, delta=2.0):
        self.n_keys = n_keys
        self.watermarks = []
        self.detectors = []
        for i in range(n_keys):
            wm = KGWWatermark(vocab_size=vocab_size, gamma=gamma, delta=delta, hash_key=42 + i * 100)
            self.watermarks.append(wm)
            self.detectors.append(KGWDetector(wm))

    def generate(self, initial_tokens, length=40):
        return self.watermarks[0].generate(initial_tokens, length=length)

    def detect(self, tokens):
        max_z = -float('inf')
        best_key = -1
        z_scores = []
        for i, detector in enumerate(self.detectors):
            z, _ = detector.detect(tokens)
            z_scores.append(z)
            if z > max_z:
                max_z = z
                best_key = i
        return max_z, best_key, z_scores

vocab_size = 500
attacker = WatermarkAttacker(vocab_size=vocab_size)
defense = MultiKeyDefense(vocab_size=vocab_size, n_keys=5)

print('=== Watermark Attack and Defense ===')
initial = torch.tensor([10, 20])
wm_tokens = defense.generate(initial, length=40)

z_orig, key_orig, z_all = defense.detect(wm_tokens)
print(f'Original: z-score={z_orig:.3f} (key={key_orig})')
z_scores_str = ', '.join(f'{z:.2f}' for z in z_all)
print(f'All keys z-scores: [{z_scores_str}]')

print(f'\nAttack robustness (single key vs multi-key):')
attacks = [
    ('Paraphrase 30%', lambda t: attacker.paraphrase_attack(t, 0.3)),
    ('Token sub 20%', lambda t: attacker.token_substitution(t, 0.2)),
    ('Token sub 40%', lambda t: attacker.token_substitution(t, 0.4)),
    ('Insert/Del 10%', lambda t: attacker.insertion_deletion(t, 0.1)),
]

for name, attack_fn in attacks:
    attacked = attack_fn(wm_tokens)
    z_single, _ = defense.detectors[0].detect(attacked)
    z_multi, best_key, _ = defense.detect(attacked)
    print(f'  {name}: single_z={z_single:.3f}, multi_z={z_multi:.3f} (key={best_key})')

print(f'\nMulti-key defense advantage:')
single_detected = 0
multi_detected = 0
n_trials = 10
for _ in range(n_trials):
    tokens = defense.generate(initial, length=40)
    attacked = attacker.token_substitution(tokens, 0.3)
    z_single, _ = defense.detectors[0].detect(attacked)
    z_multi, _, _ = defense.detect(attacked)
    if z_single > 4.0:
        single_detected += 1
    if z_multi > 4.0:
        multi_detected += 1
print(f'  Single-key detection rate: {single_detected/n_trials:.1%}')
print(f'  Multi-key detection rate: {multi_detected/n_trials:.1%}')

print(f'\nKey: Multi-key defense uses multiple watermark keys for robustness.')
print(f'Even if one key is broken, others may survive for detection.')
print(f'Stronger attacks (paraphrase) are harder to defend against than token substitution.')

## 📝 课后思考题

1. KGW水印的绿名单/红名单机制是如何工作的？为什么需要根据前一个token的哈希来动态生成绿名单？
2. 语义水印相比KGW水印在鲁棒性方面有哪些优势？这种优势的代价是什么？
3. 训练时水印与生成时水印的适用场景有何不同？在什么情况下你会选择训练时水印？
4. 面对paraphrase攻击，多密钥防御为何能提供更好的保护？还有哪些其他防御策略可以探索？

---
> 本节涵盖了模型水印与生成内容检测的核心技术。水印技术是AIGC治理的重要工具，需要在质量、可检测性和鲁棒性之间找到平衡。